# Evidências do streaming

Este notebook reúne, sobre o que o replay deixou no BigQuery, as evidências do streaming para os slides:

1. **Gold x Silver:** os agregados por janela gravados pelo Dataflow batem com a mesma conta feita em SQL sobre a Silver.
2. **Valores ausentes:** focos sem FRP (e sem as outras colunas numéricas) por satélite, depois da limpeza.
3. **Erros:** mensagens rejeitadas pela validação, por motivo, com um exemplo de cada.
4. **Janelas de 1 hora:** as janelas seguem o horário da detecção, não o da chegada.
5. **Latência:** tempo entre a publicação no Pub/Sub e a gravação na Silver, por minuto.

As consultas ficam em `sql/evidence/`, uma por seção. São as mesmas que `bash sql/run_sql.sh sql/evidence/*.sql` roda no terminal; o notebook só lê o resultado, desenha os gráficos e escreve o resumo. O grafo do job e o custo não saem de consulta: capture no console, em Dataflow > Jobs.

**Pré-requisitos:** replay completo publicado (`ingestion/run.sh replay_pubsub`) e o job já drenado, ou pelo menos 2 minutos depois do fim do replay, para a última janela ter sido gravada.

**Onde rodar:**
- **BigQuery Studio** (recomendado): no console, BigQuery > Notebook > Importar notebook. A autenticação já vem pronta. Como o notebook roda fora do repositório, ele clona o repositório público para ler `config.env` e `sql/evidence/`; por isso as consultas precisam estar no GitHub antes.
- **Na máquina:** `pip install -r notebooks/requirements.txt` e `gcloud auth application-default login`. Rodando de dentro do repositório, o notebook usa os arquivos locais.

**Custo:** antes de cada consulta o notebook estima os bytes lidos (dry run) e se recusa a rodar acima de `LIMITE_GB_POR_CONSULTA`. As tabelas do streaming têm poucos MB. No BigQuery Studio, o ambiente de execução do notebook é cobrado enquanto fica ligado; encerre-o ao terminar.

**Saídas:** gráficos (PNG), tabelas (CSV) e o resumo (`streaming_evidence.md`). Cada execução fica numa pasta própria, com a data e a hora (UTC) no nome, em `streaming_output/` e no bucket, em `evidencias/streaming/`. Rodar de novo não apaga nem sobrescreve as execuções anteriores.

## 0. Configuração

Todas as constantes ficam nesta célula. Se `../config.env` existir (notebook aberto dentro do repositório), os arquivos vêm do próprio repositório; senão (BigQuery Studio), o notebook clona o repositório público em `PASTA_CLONE` e lê dali. As consultas usam os marcadores `{{DATASET}}` e `{{DATA_CORTE}}`, trocados pelos valores de `config.env` como faz `sql/run_sql.sh`.

In [ ]:
import os
import re
import subprocess
from datetime import datetime, timezone
from typing import Any

URL_REPOSITORIO = "https://github.com/vitor-rodovalho/inpe-fire-hotspots.git"
RAMO = "main"
PASTA_CLONE = "inpe-fire-hotspots"   # Onde o repositório é clonado quando o notebook roda fora dele


def localizar_repositorio() -> str:
    """Pasta do repositório: a de cima, se o notebook estiver dentro dele; senão, um clone atualizado."""
    if os.path.exists(os.path.join("..", "config.env")):
        return ".."
    if os.path.isdir(os.path.join(PASTA_CLONE, ".git")):
        comando = ["git", "-C", PASTA_CLONE, "pull", "--ff-only", "--quiet"]
    else:
        comando = ["git", "clone", "--depth", "1", "--branch", RAMO, "--quiet", URL_REPOSITORIO, PASTA_CLONE]
    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0:
        raise RuntimeError(f"Não foi possível obter o repositório ({' '.join(comando[:3])}): {resultado.stderr.strip()}")
    return PASTA_CLONE


def carregar_config_env(caminho) -> dict[Any, Any]:
    """Lê as linhas CHAVE=VALOR do config.env do repositório, expandindo ${VAR}."""
    valores = {}
    with open(caminho, encoding="utf-8") as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            if not linha or linha.startswith("#") or "=" not in linha:
                continue
            chave, valor = linha.split("=", 1)
            valor = re.split(r"\s+#", valor, maxsplit=1)[0].strip().strip('"')
            valor = re.sub(r"\$\{(\w+)\}", lambda m: valores.get(m.group(1), ""), valor)
            valores[chave.strip()] = valor
    return valores


PASTA_REPOSITORIO = localizar_repositorio()
CONFIG = carregar_config_env(os.path.join(PASTA_REPOSITORIO, "config.env"))

# =============================================================================
# Constantes (vindas de config.env)
# =============================================================================
PROJECT_ID = CONFIG["PROJECT_ID"]
REGIAO = CONFIG["REGIAO"]
DATASET = CONFIG["DATASET"]
BUCKET = CONFIG["BUCKET"]
DATA_CORTE = CONFIG["DATA_CORTE"]        # Primeiro dia do streaming

PASTA_SQL = os.path.join(PASTA_REPOSITORIO, "sql", "evidence")
LIMITE_GB_POR_CONSULTA = 1               # Nenhuma consulta roda se for ler mais do que isso
EXECUCAO = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")   # Nome da pasta desta execução (UTC)
PASTA_SAIDA = os.path.join("streaming_output", EXECUCAO)            # Gráficos (PNG), tabelas (CSV) e resumo (Markdown)
ENVIAR_PARA_BUCKET = True                                           # Copia as saídas para o bucket no fim
PASTA_BUCKET_SAIDA = f"evidencias/streaming/{EXECUCAO}"

FORMATO_DATA = "%d/%m/%Y"
FORMATO_DATA_HORA = "%d/%m %H:%M"

# Cores dos gráficos (as mesmas do notebook 01)
COR_PRINCIPAL = "#2a78d6"
COR_SECUNDARIA = "#eb6834"
COR_NEUTRA = "#a3a29c"
COR_REFERENCIA = "#52514e"
# =============================================================================

print(f"Repositório: {os.path.abspath(PASTA_REPOSITORIO)}")
print(f"Projeto: {PROJECT_ID} | região: {REGIAO} | dataset: {DATASET} | streaming a partir de {DATA_CORTE}")
print(f"Consultas: {sorted(os.listdir(PASTA_SQL))}")
print(f"Saídas desta execução: {PASTA_SAIDA}" + (f" e gs://{BUCKET}/{PASTA_BUCKET_SAIDA}/" if ENVIAR_PARA_BUCKET else ""))

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from google.cloud import bigquery, storage

cliente = bigquery.Client(project=PROJECT_ID, location=REGIAO)
os.makedirs(PASTA_SAIDA, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 200)

BYTES_LIDOS = {"total": 0}
RESUMO = {}   # Números-chave, preenchidos ao longo do notebook
MARCADOR = re.compile(r"\{\{(\w+)\}\}")

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.6,
})


def ler_sql(nome) -> str:
    """Lê um arquivo de sql/evidence e troca os marcadores {{CHAVE}} pelos valores de config.env."""
    with open(os.path.join(PASTA_SQL, nome), encoding="utf-8") as arquivo:
        texto = arquivo.read()
    faltando = sorted(set(MARCADOR.findall(texto)) - set(CONFIG))
    if faltando:
        raise KeyError(f"{nome} usa marcadores que não estão em config.env: {faltando}")
    return MARCADOR.sub(lambda m: CONFIG[m.group(1)], texto)


def consultar(sql) -> Any:
    """Roda uma consulta com estimativa de custo antes (dry run) e limite de bytes."""
    limite = int(LIMITE_GB_POR_CONSULTA * 1024**3)

    def configuracao(dry_run) -> bigquery.QueryJobConfig:
        return bigquery.QueryJobConfig(
            dry_run=dry_run,
            use_query_cache=not dry_run,
            maximum_bytes_billed=None if dry_run else limite,
        )

    estimativa = cliente.query(sql, job_config=configuracao(True)).total_bytes_processed or 0
    if estimativa > limite:
        raise RuntimeError(
            f"Consulta leria {estimativa / 1024**3:.2f} GB, acima do limite de {LIMITE_GB_POR_CONSULTA} GB."
        )
    job = cliente.query(sql, job_config=configuracao(False))
    resultado = job.to_dataframe()
    lidos = job.total_bytes_processed or 0
    BYTES_LIDOS["total"] += lidos
    print(f"[{lidos / 1024**2:,.1f} MB lidos, {BYTES_LIDOS['total'] / 1024**3:,.2f} GB no notebook]")
    return resultado


def consultar_arquivo(nome) -> Any:
    """Roda um arquivo de sql/evidence e guarda o resultado em CSV, com o mesmo nome."""
    resultado = consultar(ler_sql(nome))
    caminho = os.path.join(PASTA_SAIDA, nome.replace(".sql", ".csv"))
    resultado.to_csv(caminho, index=False)
    print(f"Tabela salva em {caminho}")
    return resultado


def salvar(figura, nome) -> None:
    caminho = os.path.join(PASTA_SAIDA, f"{nome}.png")
    figura.savefig(caminho, dpi=150, bbox_inches="tight")
    print(f"Gráfico salvo em {caminho}")


def milhar(valor, _posicao=None) -> str:
    return f"{valor:,.0f}".replace(",", ".")


def decimal(valor, casas=1) -> str:
    """Número com vírgula decimal e ponto de milhar, como nos slides."""
    return f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def lista(valores, vazio="nenhum") -> str:
    valores = list(valores)
    return ", ".join(str(v) for v in valores) if valores else vazio

## 1. Gold x Silver

Faz a mesma conta da `gold_agregados_janela` sobre a Silver de streaming (focos e focos sem FRP por hora de detecção e estado) e resume as janelas por situação. O esperado é uma única linha `ok`, com o mesmo número de focos dos dois lados. `max_gravacoes_por_janela` maior que 1 é normal: a janela é gravada de novo a cada leva de focos atrasados, e vale a gravação mais recente.

In [ ]:
gold_vs_silver = consultar_arquivo("01_gold_vs_silver.sql")
if gold_vs_silver.empty:
    raise RuntimeError("As tabelas do streaming estão vazias. Publique o replay antes de rodar este notebook.")

janelas_por_situacao = dict(zip(gold_vs_silver["situacao"], gold_vs_silver["janelas"].astype(int)))
RESUMO["janelas_total"] = sum(janelas_por_situacao.values())
RESUMO["janelas_ok"] = janelas_por_situacao.get("ok", 0)
RESUMO["janelas_fora_do_ok"] = {s: n for s, n in janelas_por_situacao.items() if s != "ok"}
RESUMO["focos_gold"] = int(gold_vs_silver["focos_na_gold"].sum())
RESUMO["focos_silver"] = int(gold_vs_silver["focos_na_silver"].sum())
RESUMO["max_gravacoes"] = int(gold_vs_silver["max_gravacoes_por_janela"].fillna(0).max())

if not RESUMO["janelas_fora_do_ok"]:
    print(f"OK: as {milhar(RESUMO['janelas_total'])} janelas da gold batem com a Silver "
          f"({milhar(RESUMO['focos_gold'])} focos dos dois lados).")
else:
    print(f"ATENÇÃO: janelas fora de 'ok': {RESUMO['janelas_fora_do_ok']}")
    print("  'falta_na_gold' logo depois do replay costuma ser a última hora, ainda não gravada: espere 2 minutos e rode de novo.")
    print("  Para ver janela por janela, rode sql/demo/04_check_aggregates.sql.")
gold_vs_silver

## 2. Valores ausentes

Na limpeza, vazio e `-999` viram nulo em todas as colunas numéricas (`para_float` em `pipeline/focos_stream/cleaning.py`). A primeira linha (`TODOS`) é o total; as outras, um satélite cada. Os agregados guardam os focos sem FRP à parte, na coluna `qtd_sem_frp`.

In [ ]:
ausentes = consultar_arquivo("02_missing_values.sql")
total = ausentes.loc[ausentes["nome_satelite"] == "TODOS"].iloc[0]
por_satelite = ausentes.loc[ausentes["nome_satelite"] != "TODOS"].copy()

RESUMO["focos_streaming"] = int(total["focos"])
RESUMO["sem_frp"] = int(total["sem_frp"])
RESUMO["pct_sem_frp"] = float(total["pct_sem_frp"])
RESUMO["satelites"] = len(por_satelite)
RESUMO["satelites_nunca_tem_frp"] = por_satelite.loc[por_satelite["sem_frp"] == por_satelite["focos"], "nome_satelite"].tolist()
RESUMO["outros_ausentes"] = {
    "risco de fogo": int(total["sem_risco_fogo"]),
    "dias sem chuva": int(total["sem_dias_sem_chuva"]),
    "precipitação": int(total["sem_precipitacao"]),
}
print(f"{milhar(RESUMO['focos_streaming'])} focos, {decimal(RESUMO['pct_sem_frp'])}% sem FRP, {RESUMO['satelites']} satélites")
ausentes

In [ ]:
por_satelite = por_satelite.sort_values("pct_sem_frp")
figura, eixo = plt.subplots(figsize=(9, 0.38 * len(por_satelite) + 1.2))
eixo.barh(por_satelite["nome_satelite"], por_satelite["pct_sem_frp"], color=COR_PRINCIPAL, height=0.65)
eixo.axvline(RESUMO["pct_sem_frp"], color=COR_REFERENCIA, linestyle="--", linewidth=1.2,
             label=f"todos os satélites: {decimal(RESUMO['pct_sem_frp'])}%")
for posicao, (pct, focos) in enumerate(zip(por_satelite["pct_sem_frp"], por_satelite["focos"])):
    eixo.annotate(f"{decimal(pct)}% de {milhar(focos)}", (pct, posicao), xytext=(4, 0), textcoords="offset points",
                  va="center", fontsize=9)
eixo.legend(frameon=False, loc="lower right")
eixo.set_xlim(0, 115)
eixo.xaxis.set_major_formatter(mticker.PercentFormatter())
eixo.grid(axis="y", visible=False)
eixo.set_title("Focos sem FRP por satélite no streaming")
eixo.set_xlabel("% dos focos do satélite")
salvar(figura, "02_sem_frp_por_satelite")
plt.show()

## 3. Erros

Mensagens rejeitadas pela validação, por motivo. No replay, essas mensagens são cópias estragadas de propósito (`--fracao-invalidas`); o foco original segue intacto para a Silver. A porcentagem é sobre todas as mensagens lidas (focos aceitos mais rejeitadas).

In [ ]:
erros = consultar_arquivo("03_errors_by_reason.sql")
RESUMO["erros_total"] = int(erros["mensagens"].sum())
RESUMO["erros_por_motivo"] = dict(zip(erros["motivo"], erros["mensagens"].astype(int)))
mensagens_lidas = RESUMO["erros_total"] + RESUMO["focos_streaming"]
RESUMO["pct_erros"] = round(100 * RESUMO["erros_total"] / mensagens_lidas, 2) if mensagens_lidas else 0.0
print(f"{milhar(RESUMO['erros_total'])} mensagens rejeitadas, {decimal(RESUMO['pct_erros'], 2)}% das mensagens lidas")
erros

In [ ]:
if not erros.empty:
    ordenado = erros.sort_values("mensagens")
    figura, eixo = plt.subplots(figsize=(9, 0.5 * len(ordenado) + 1.2))
    eixo.barh(ordenado["motivo"], ordenado["mensagens"], color=COR_SECUNDARIA, height=0.6)
    for posicao, quantidade in enumerate(ordenado["mensagens"]):
        eixo.annotate(milhar(quantidade), (quantidade, posicao), xytext=(4, 0), textcoords="offset points", va="center")
    eixo.set_xlim(0, ordenado["mensagens"].max() * 1.15)
    eixo.xaxis.set_major_formatter(mticker.FuncFormatter(milhar))
    eixo.grid(axis="y", visible=False)
    eixo.set_title(f"Mensagens rejeitadas pela validação ({decimal(RESUMO['pct_erros'], 2)}% das mensagens)")
    eixo.set_xlabel("mensagens")
    salvar(figura, "03_erros_por_motivo")
    plt.show()

## 4. Janelas de 1 hora pelo horário da detecção

A consulta escolhe o estado e o dia com mais focos e mostra as janelas desse dia, uma por hora. Cada janela é uma hora real de fogo daquele dia, mas `calculado_em` é o momento do replay: a janela vem do horário da detecção, não do horário em que o foco chegou. A seção 1 mostra que as contas estão certas.

In [ ]:
janelas_dia = consultar_arquivo("04_windows_one_day.sql")
dia = janelas_dia["inicio_janela"].min()
RESUMO["janelas_estado"] = janelas_dia["estado"].iloc[0]
RESUMO["janelas_dia"] = dia.strftime(FORMATO_DATA)
RESUMO["janelas_qtd"] = len(janelas_dia)
RESUMO["janelas_focos"] = int(janelas_dia["qtd_focos"].sum())
RESUMO["calculado_de"] = janelas_dia["calculado_em"].min().strftime(FORMATO_DATA_HORA)
RESUMO["calculado_ate"] = janelas_dia["calculado_em"].max().strftime(FORMATO_DATA_HORA)
print(f"{RESUMO['janelas_estado']} em {RESUMO['janelas_dia']}: {RESUMO['janelas_qtd']} janelas, "
      f"{milhar(RESUMO['janelas_focos'])} focos, calculadas entre {RESUMO['calculado_de']} e {RESUMO['calculado_ate']} UTC")
janelas_dia

In [ ]:
horas = janelas_dia["inicio_janela"].dt.hour
com_frp = janelas_dia["qtd_focos"] - janelas_dia["qtd_sem_frp"]
figura, eixo = plt.subplots(figsize=(10, 4))
eixo.bar(horas, com_frp, color=COR_PRINCIPAL, width=0.75, label="com FRP")
eixo.bar(horas, janelas_dia["qtd_sem_frp"], bottom=com_frp, color=COR_NEUTRA, width=0.75, label="sem FRP")
eixo.set_xticks(range(0, 24))
eixo.set_xlim(-0.7, 23.7)
eixo.yaxis.set_major_formatter(mticker.FuncFormatter(milhar))
eixo.grid(axis="x", visible=False)
eixo.legend(frameon=False, loc="upper left")
eixo.set_title(f"Focos por janela de 1 hora: {RESUMO['janelas_estado']}, {RESUMO['janelas_dia']}")
eixo.set_xlabel("hora da detecção (UTC)")
eixo.set_ylabel("focos")
figura.text(0.01, -0.04, f"Janelas calculadas pelo Dataflow entre {RESUMO['calculado_de']} e "
            f"{RESUMO['calculado_ate']} UTC, durante o replay.", color=COR_REFERENCIA, ha="left")
salvar(figura, "04_janelas_um_dia")
plt.show()

## 5. Latência

Tempo entre o replay publicar o foco no Pub/Sub (`publicado_em`) e o Dataflow gravá-lo na Silver (`ingerido_em`), por minuto de publicação, em UTC (3 horas à frente de Brasília). A primeira linha da consulta, com o minuto vazio, é o total. Com o job já em execução, a mediana fica em poucos segundos; nos minutos em que o job ainda subia, os focos esperaram na fila.

In [ ]:
latencia = consultar_arquivo("05_latency_by_minute.sql")
total_latencia = latencia.loc[latencia["minuto_publicacao_utc"].isna()].iloc[0]
por_minuto = latencia.loc[latencia["minuto_publicacao_utc"].notna()].copy()

RESUMO["latencia_focos"] = int(total_latencia["focos"])
RESUMO["latencia_p50"] = float(total_latencia["p50_s"])
RESUMO["latencia_p95"] = float(total_latencia["p95_s"])
RESUMO["latencia_max"] = float(total_latencia["max_s"])
RESUMO["latencia_minutos"] = len(por_minuto)
RESUMO["latencia_p50_menor"] = float(por_minuto["p50_s"].min())
RESUMO["latencia_p50_maior"] = float(por_minuto["p50_s"].max())
RESUMO["latencia_de"] = por_minuto["minuto_publicacao_utc"].min().strftime(FORMATO_DATA_HORA)
RESUMO["latencia_ate"] = por_minuto["minuto_publicacao_utc"].max().strftime(FORMATO_DATA_HORA)
print(f"Total: mediana {decimal(RESUMO['latencia_p50'], 2)} s, p95 {decimal(RESUMO['latencia_p95'], 2)} s, "
      f"máximo {decimal(RESUMO['latencia_max'], 2)} s "
      f"em {milhar(RESUMO['latencia_focos'])} focos e {RESUMO['latencia_minutos']} minutos de publicação")
latencia

In [ ]:
figura, eixo = plt.subplots(figsize=(10, 4))
minutos = por_minuto["minuto_publicacao_utc"]
eixo.plot(minutos, por_minuto["p50_s"], color=COR_PRINCIPAL, linewidth=2, marker="o", markersize=3, label="mediana (p50)")
eixo.plot(minutos, por_minuto["p95_s"], color=COR_SECUNDARIA, linewidth=1.5, marker="o", markersize=3, label="p95")
if (por_minuto[["p50_s", "p95_s"]] > 0).all().all():
    eixo.set_yscale("log")
    eixo.yaxis.set_major_locator(mticker.LogLocator(subs=(1.0, 2.0, 5.0)))
    eixo.yaxis.set_major_formatter(mticker.FuncFormatter(lambda valor, _posicao: f"{valor:g}".replace(".", ",")))
    eixo.yaxis.set_minor_formatter(mticker.NullFormatter())
eixo.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
eixo.legend(frameon=False, loc="upper right")
eixo.set_title("Latência da publicação no Pub/Sub até a gravação na Silver, por minuto")
eixo.set_xlabel(f"minuto da publicação (UTC), de {RESUMO['latencia_de']} a {RESUMO['latencia_ate']}")
eixo.set_ylabel("segundos")
salvar(figura, "05_latencia_por_minuto")
plt.show()

## 6. Resumo e cópia para o bucket

Gera o resumo em Markdown com os números-chave, salvo como `streaming_evidence.md` na pasta desta execução, junto com os gráficos e as tabelas. Depois copia a pasta para o bucket, em `gs://<BUCKET>/evidencias/streaming/<execução>/`, para baixar pelo console ou passar para quem monta os slides. No BigQuery Studio, os arquivos também aparecem no painel de arquivos do notebook.

In [ ]:
gerado_em = datetime.now(timezone.utc).strftime("%d/%m/%Y %H:%M UTC")
linhas = [
    "# Evidências do streaming",
    "",
    f"Gerado pelo notebook `notebooks/02_streaming_evidence.ipynb` em {gerado_em}, sobre o dataset "
    f"`{DATASET}` a partir de {DATA_CORTE}. As consultas estão em `sql/evidence/`.",
    "",
    "## Gold x Silver",
    f"- {milhar(RESUMO['janelas_ok'])} de {milhar(RESUMO['janelas_total'])} janelas batem (situação ok); "
    f"focos: {milhar(RESUMO['focos_gold'])} na gold e {milhar(RESUMO['focos_silver'])} na Silver.",
    f"- Janelas fora de ok: {lista((f'{s} ({n})' for s, n in RESUMO['janelas_fora_do_ok'].items()), 'nenhuma')}.",
    f"- Até {RESUMO['max_gravacoes']} gravações por janela e estado (a janela é gravada de novo quando "
    "chegam focos atrasados; vale a mais recente).",
    "",
    "## Valores ausentes",
    f"- {milhar(RESUMO['focos_streaming'])} focos de {RESUMO['satelites']} satélites; "
    f"{decimal(RESUMO['pct_sem_frp'])}% sem FRP ({milhar(RESUMO['sem_frp'])}).",
    f"- Satélites que nunca trazem FRP: {lista(RESUMO['satelites_nunca_tem_frp'])}.",
    f"- Outras colunas sem valor: {lista(f'{c} ({milhar(n)})' for c, n in RESUMO['outros_ausentes'].items())}.",
    "",
    "## Erros",
    f"- {milhar(RESUMO['erros_total'])} mensagens rejeitadas ({decimal(RESUMO['pct_erros'], 2)}% das mensagens lidas): "
    f"{lista(f'{m} ({milhar(n)})' for m, n in RESUMO['erros_por_motivo'].items())}.",
    "- São cópias estragadas de propósito pelo replay (--fracao-invalidas); o foco original segue para a Silver.",
    "",
    "## Janelas de 1 hora",
    f"- Estado e dia com mais focos: {RESUMO['janelas_estado']}, {RESUMO['janelas_dia']}, com "
    f"{RESUMO['janelas_qtd']} janelas e {milhar(RESUMO['janelas_focos'])} focos.",
    f"- Janelas pelo horário da detecção, calculadas entre {RESUMO['calculado_de']} e "
    f"{RESUMO['calculado_ate']} UTC, durante o replay.",
    "",
    "## Latência (publicação no Pub/Sub até a gravação na Silver)",
    f"- Mediana {decimal(RESUMO['latencia_p50'], 2)} s, p95 {decimal(RESUMO['latencia_p95'], 2)} s e "
    f"máximo {decimal(RESUMO['latencia_max'], 2)} s, "
    f"em {milhar(RESUMO['latencia_focos'])} focos.",
    f"- Mediana por minuto entre {decimal(RESUMO['latencia_p50_menor'], 2)} s e "
    f"{decimal(RESUMO['latencia_p50_maior'], 2)} s, em "
    f"{RESUMO['latencia_minutos']} minutos de publicação ({RESUMO['latencia_de']} a {RESUMO['latencia_ate']} UTC).",
    "",
    f"Total lido pelo notebook: {decimal(BYTES_LIDOS['total'] / 1024**3, 2)} GB.",
]
if ENVIAR_PARA_BUCKET:
    linhas.append(f"Arquivos desta execução: `gs://{BUCKET}/{PASTA_BUCKET_SAIDA}/`.")

texto = "\n".join(linhas)
with open(os.path.join(PASTA_SAIDA, "streaming_evidence.md"), "w", encoding="utf-8") as arquivo:
    arquivo.write(texto + "\n")
print(texto)

In [ ]:
if ENVIAR_PARA_BUCKET:
    balde = storage.Client(project=PROJECT_ID).bucket(BUCKET)
    arquivos = sorted(os.listdir(PASTA_SAIDA))
    for nome in arquivos:
        balde.blob(f"{PASTA_BUCKET_SAIDA}/{nome}").upload_from_filename(os.path.join(PASTA_SAIDA, nome))
    print(f"{len(arquivos)} arquivos copiados para gs://{BUCKET}/{PASTA_BUCKET_SAIDA}/")
    for nome in arquivos:
        print(f"  {nome}")